# TO DO LIST

* scrivi conclusioni sui tagli pair variables
* usa più dati (salvando valori degli istogrammi, spostando il cut iniziale per le single variabs a monte nella parte di upload, cosi si salva ram...)
* prova diverse funzioni di bg nel fit di massa invariante: magari prova ciascuno per ogni fascia di pt, se c'è una funzione di fondo che è milgiore nella maggior parte delle fasce, teniamo quello

Aggiunte eventuali:
* altri cut combinati di pair variables? tipo cos poynting x decay length...
* nei grafici di pair variables, unire quelli di D0 e anti-D0 in uno singolo di segnale (tanto sono uguali) cosi si guadagna in chiarezza?
* riprovare con fit quadratico per background quando fittiamo la massa invariante

ALREADY DONE:

* usa tutti dati MC grezzi per notebook MC single tracks
* sistema binning grafici
* scrivi conclusioni sui tagli single variables
* includi dati a TOF=-999 e cerca best cut su TPC
* nella ricerca dei best cut aggiungi come parametro di search la pseudo-significance: S / sqrt(B) ---> risultato: non cambia nulla, il BG è troppo grande rispetto al segnale quindi sqrt(S+B)=sqrt(B)
* separazione D0 e anti-D0: casi in cui tengo una sola ipotesi e casi in cui tengo entrambe

In [ ]:
# --- Initial parameter guesses ---
k_param = 100
gauss_params = [m_d0, 0.05]

# ------------------------------------------------------------------
# Histogram of TRUE signal and TRUE reflected components

# HISTOGRAM:
fig, ax = plt.subplots(figsize=(13,6))
# ------- fake component:
fake_counts, fake_edges, _ = ax.hist(mass_bg_all_but, bins=N_BINS, range=(MIN_MASS, MAX_MASS), alpha=0.5, label='fake', color='orange')
fake_centers = (fake_edges[:-1] + fake_edges[1:]) / 2.0
# poisson errors:
sigma_fake_hist = np.sqrt(fake_counts)
sigma_fake_hist = np.clip(sigma_fake_hist, 1, None)  # avoid zeros
# ------- signal:
sign_counts, sign_edges, _ = ax.hist(signal_data, bins=N_BINS, range=(MIN_MASS, MAX_MASS), alpha=0.5, label='Signal', color='green')
sign_centers = (sign_edges[:-1] + sign_edges[1:]) / 2.0
# poisson errors:
sigma_sign_hist = np.sqrt(sign_counts)
sigma_sign_hist = np.clip(sigma_sign_hist, 1, None)  # avoid zeros

# FIT:
sign_fit_res = curve_fit( fit_gauss_mc, xdata = sign_centers, ydata = sign_counts, sigma = sigma_sign_hist,
                            p0 = [k_param, 
                                  gauss_params[0], gauss_params[1]
                                 ]
                        )
sign_params = sign_fit_res[0]
mu_sign, sigma_sign = sign_params[1], sign_params[2]
k_sign = sign_params[0]
# ----
fake_fit_res = curve_fit( fit_gauss_mc, xdata = fake_centers, ydata = fake_counts, sigma = sigma_fake_hist,
                            p0 = [k_param, 
                                  gauss_params[0], gauss_params[1]
                                 ]
                        )
fake_params = fake_fit_res[0]
mu_fake, sigma_fake = fake_params[1], fake_params[2]
k_fake = fake_params[0]

# PLOT of fitted curves:
x_fit = np.linspace(MIN_MASS, MAX_MASS, 1000)
ax.plot(x_fit, fit_gauss_mc(x_fit, k_fake, mu_fake, sigma_fake), '--', color='darkorange', linewidth=2, label='fake Fit')
ax.plot(x_fit, fit_gauss_mc(x_fit, k_sign, mu_sign, sigma_sign), '--', color='darkgreen', linewidth=2, label='Signal Fit')


ax.set_xlabel("Invariant Mass [GeV/c²]")
ax.set_ylabel(f"Counts per {binsize*1000:.2f} MeV/c²")
ax.set_title("Fit of Signal + Reflected using MC info")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

# PRINT:
ratio_from_fit = k_fake/k_sign
print("\n================ FIT RESULTS =================")

print(f"{'Fit':<12} {'mu':>12} {'sigma':>12} {'k value':>12}")
print("-"*48)
print(f"{'Signal':<12} {mu_sign:>12.4f} {sigma_sign:>12.4f} {k_sign:>12.3f}")
print(f"{'fake':<12} {mu_fake:>12.4f} {sigma_fake:>12.4f} {k_fake:>12.3f} \n")

print(f"Ratio N1/N2 (fake/signal):   {ratio_from_fit:>20.3f} ")